# Info 2950 Project- Phase 3 Data Cleaning
Members:

Marianna Ruggiero
mir46@cornell.edu

Joyce Han
jh2969@cornell.edu

Fiona Chandler
fcc42@cornell.edu

Friana Engineer
fe63@cornell.edu

Research Questions: How do the technical skills that employers request differ between US data analyst, data scientist and data engineer job postings in 2023?how often are those skills mentioned in Cornells computing and data related course descriptions from the same year?

## Raw Data Sources

- **Cornell courses:** We use the [saved Cornell Class Roster JSON files](https://github.com/FrianaEngineer/INFO-2950-Group-Project/tree/main/phase2/raw) for spring and fall 2023. The files are in phase2/raw. We use 4 cornell files from spring 2023 and 4 from fall 2023. Each semester has one file for each subject- CS, INFO, STSCI and ORIE.

- **Job postings:** Download the `data_jobs.csv` from the [Hugging Face dataset files](https://huggingface.co/datasets/lukebarousse/data_jobs/tree/main) and save it in `phase3/` in the same folder as this notebook.

This notebook uses those files to create cornell_courses_2023_clean.csv and job_postings_2023_clean.csv for the phase 3 report.

In [37]:
# Read the eight saved Cornell files and combine them into one table.
import json
from pathlib import Path

import pandas as pd

raw_folder = Path("../phase2/raw")
terms = ["SP23", "FA23"]
subjects = ["CS", "INFO", "STSCI", "ORIE"]

raw_files = []
for term in terms:
    for subject in subjects:
        raw_files.append(raw_folder / f"{term}_{subject}.json")

rows = []
for path in sorted(raw_files):
    term = path.stem.split("_")[0]
    data = json.loads(path.read_text())

    for course in data["data"]["classes"]:
        rows.append({
            "term": term,
            "crseId": course.get("crseId"),
            "subject": course.get("subject"),
            "catalogNbr": course.get("catalogNbr"),
            "title": course.get("titleLong") or "",
            "description": course.get("description") or "",
        })

df = pd.DataFrame(rows)

## Clean the Cornell course data

We keep courses with a description and a course ID. We need the description to look for skill mentions and the ID to count each course only once. If a course has different descriptions across semesters, we keep both so we can search both without counting the course twice.

In [38]:
# Check the proposed Cornell course sample for Phase 3.
subjects = ["CS", "INFO", "STSCI", "ORIE"]
terms = ["SP23", "FA23"]

cornell_2023 = df.loc[
    df["term"].isin(terms) & df["subject"].isin(subjects)
].copy()

cornell_2023["has_description"] = (
    cornell_2023["description"].fillna("").str.strip().ne("")
)

course_check = cornell_2023.groupby(
    ["term", "subject"], observed=True
).agg(
    roster_rows=("has_description", "size"),
    rows_with_descriptions=("has_description", "sum"),
)

course_check

roster_rows  rows_with_descriptions
term subject                                     
FA23 CS                96                      92
     INFO              65                      65
     ORIE              58                      58
     STSCI             46                      45
SP23 CS               100                      98
     INFO              74                      74
     ORIE              45                      45
     STSCI             51                      50

In [39]:
# Compare roster rows, course offerings, and distinct courses.
described = cornell_2023.loc[cornell_2023["has_description"]]
identified = described.dropna(subset=["crseId"])

print("Rows with descriptions:", len(described))
print("Rows missing a course ID:", len(described) - len(identified))
print("Distinct course offerings:", len(
    identified.drop_duplicates(subset=["term", "crseId"])
))
print("Distinct courses across 2023:", identified["crseId"].nunique())

Rows with descriptions: 527
Rows missing a course ID: 0
Distinct course offerings: 494
Distinct courses across 2023: 398


In [40]:
# Check whether the same course has different descriptions in the roster.
description_counts = (
    identified.assign(
        clean_description=identified["description"]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    .groupby("crseId")["clean_description"]
    .nunique()
)

print("Courses with multiple descriptions:", (description_counts > 1).sum())

Courses with multiple descriptions: 7


In [41]:
# Show the roster entries for courses with multiple descriptions.
multiple_ids = description_counts[description_counts > 1].index

description_review = identified.loc[
    identified["crseId"].isin(multiple_ids),
    ["crseId", "term", "subject", "catalogNbr", "title", "description"]
].sort_values(["crseId", "term", "subject"])

with pd.option_context("display.max_colwidth", None):
    display(description_review)

,crseId,term,subject,catalogNbr,title,description
231,352287,FA23,STSCI,4090,Theory of Statistics,"Introduction to classical theory of parametric statistical inference that builds on the material covered in BTRY 3080. Topics include: sampling distributions, principles of data reduction, likelihood, parameter estimation, hypothesis testing, interval estimation, and basic asymptotic theory."
495,352287,SP23,STSCI,4090,Theory of Statistics,"Introduction to classical theory of parametric statistical inference that builds on the material covered in BTRY 3080. Topics include sampling distributions, principles of data reduction, likelihood, parameter estimation, hypothesis testing, interval estimation, and basic asymptotic theory."
1,358525,FA23,CS,1112,Introduction to Computing: An Engineering and Science Perspective,"Programming and problem solving using Python. Emphasizes the systematic development of algorithms and programs. Topics include iteration, functions, arrays, strings, recursion, object-oriented programming, algorithms, and data handling and visualization. Assignments are designed to build an appreciation for complexity, dimension, randomness, simulation, and the role of approximation in engineering and science. Weekly discussion section provides guided practice on the computer, with staff present to help. NO programming experience is necessary; some knowledge of Calculus is required."
266,358525,SP23,CS,1112,Introduction to Computing Using MATLAB,"Programming and problem solving using MATLAB. Emphasizes the systematic development of algorithms and programs. Topics include iteration, functions, arrays, recursion, object-oriented programming, and MATLAB graphics. Assignments are designed to build an appreciation for complexity, dimension, fuzzy data, inexact arithmetic, randomness, simulation, and the role of approximation."
102,358808,FA23,INFO,2950,Introduction to Data Science,"INFO 2950 is an applied introductory course on the foundations of data science, focusing on using data to identify patterns, evaluating the strength and significance of relationships, and generating predictions using data. Topics covered include the core principles of statistical programming (such as data frames, Python/R packages, reproducible workflows, and version control), univariate and multivariate statistical analysis of small and medium-size datasets, regression methods, hypothesis testing, probability models, basic supervised and unsupervised machine learning, data visualization, and network analysis. Students will learn how to use data to make effective arguments in a way that promotes the ethical usage of data. Students who complete the course will be able to produce meaningful, data-driven analyses of real-world problems and will be prepared to begin more advanced work in data-intensive domains."
371,358808,SP23,INFO,2950,Introduction to Data Science,"Teaches basic mathematical methods for information science, with applications to data science. Topics include discrete probability, Bayesian methods, graph theory, power law distributions, Markov models, and hidden Markov models. Uses examples and applications from various areas of information science such as the structure of the web, genomics, social networks, natural language processing, and signal processing. Assignments require python programming.\n"
221,358957,FA23,STSCI,2130,Applied Regression Analysis,"This seven week, two-credit class will cover the regression requirements, hypothesis tests, and interpretation of results. Students will learn to identify the data necessary to perform a regression analysis, evaluate the conditions, and apply the statistical tests. Interpretation of overall results will be made. Independent/group projects by each student will be done. These will consist of identifying an issue of interest, a relevant data set, and analysis using the regression methods. Presentation of the results in verbal and written form will be required. Recommended for students 

In [42]:
# Keep one row per course ID and preserve every distinct description.
course_descriptions = identified[["crseId", "description"]].copy()
course_descriptions["description"] = (
    course_descriptions["description"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

courses_2023_clean = (
    course_descriptions.drop_duplicates()
    .groupby("crseId", as_index=False)["description"]
    .agg("\n---\n".join)
)

courses_2023_clean.to_csv("cornell_courses_2023_clean.csv", index=False)
print("Courses exported:", len(courses_2023_clean))

Courses exported: 398


## Clean the job-posting data

We keep 2023 US postings for data analyst, data scientist and data engineer roles. We remove exact duplicate rows and leave out postings that dont have a usable skill list. then put skill names in lowercase and remove repeats within each posting so that a posting counts only once for each skill.

In [43]:
# Check how many 2023 U.S. postings have skill information for each role.

jobs = pd.read_csv("data_jobs.csv")
roles = ["Data Analyst", "Data Scientist", "Data Engineer"]
posted_year = pd.to_datetime(jobs["job_posted_date"], errors="coerce").dt.year

jobs_2023 = jobs.loc[
    jobs["job_country"].eq("United States")
    & posted_year.eq(2023)
    & jobs["job_title_short"].isin(roles)
].copy()

skills = jobs_2023["job_skills"].fillna("").str.strip()
jobs_2023["has_skill_list"] = ~skills.isin(["", "[]"])

jobs_2023.groupby("job_title_short").agg(
    total_postings=("has_skill_list", "size"),
    postings_with_skills=("has_skill_list", "sum")
)

,total_postings,postings_with_skills
job_title_short,,
Data Analyst,67828,57413
Data Engineer,34583,32815
Data Scientist,58868,51686


In [44]:
# Remove rows that match across every original column.
jobs_2023_unique = jobs_2023.drop_duplicates().copy()

print("Exact duplicate rows removed:", len(jobs_2023) - len(jobs_2023_unique))

jobs_2023_unique.groupby("job_title_short").agg(
    total_postings=("has_skill_list", "size"),
    postings_with_skills=("has_skill_list", "sum")
)

Exact duplicate rows removed: 5


,total_postings,postings_with_skills
job_title_short,,
Data Analyst,67826,57411
Data Engineer,34582,32814
Data Scientist,58866,51685


In [45]:
import ast

# Keep postings with skill information.
jobs_clean = jobs_2023_unique.loc[
    jobs_2023_unique["has_skill_list"],
    ["job_title_short", "job_skills"]
].copy()

# Convert each list from text and count each skill once per posting.
def clean_skills(skill_text):
    skills = ast.literal_eval(skill_text)
    return sorted({skill.strip().lower() for skill in skills if skill.strip()})

jobs_clean["job_skills"] = jobs_clean["job_skills"].map(clean_skills)
jobs_clean = jobs_clean.loc[jobs_clean["job_skills"].map(len) > 0]

jobs_clean.to_csv("job_postings_2023_clean.csv", index=False)
print("Postings exported:", len(jobs_clean))
jobs_clean.head()

Postings exported: 141910


,job_title_short,job_skills
4,Data Analyst,"[jira, python, sql]"
6,Data Scientist,"[docker, python, r, splunk, tableau]"
7,Data Scientist,"[alteryx, cognos, python, r, spss, sql, tableau]"
14,Data Scientist,"[excel, python, r, sql, tableau]"
16,Data Engineer,"[azure, bash, databricks, excel, git, hadoop, ..."
